# FASE 2. EXPLORACIÓN, PREPROCESAMIENTO Y VALIDACIÓN DE DATOS

**Proyecto:** Análisis del impacto operacional documentado de las fallas del Sistema Eléctrico Nacional sobre comunas prioritarias de Chile mediante ciencia de datos.

**Asignatura:** Programación para la Ciencia de Datos (MCDI500)
**Programa:** Magíster en Ciencia de Datos e Inteligencia Artificial
**Institución:** Universidad Andrés Bello
**Grupo:** 7

---

## 1. Introducción y objetivos de la Fase 2

### 1.1. Introducción

La Fase 2 desarrolla el procesamiento inicial de los registros históricos de fallas del Sistema Eléctrico Nacional (SEN), con el propósito de establecer una base de datos consistente, reproducible y técnicamente adecuada para el análisis territorial.

El conjunto de información proviene de las publicaciones del Coordinador Eléctrico Nacional (CEN), correspondientes al período enero de 2024 a septiembre de 2026. Durante la Fase 1 se implementó un procedimiento de consolidación que permitió integrar 33 publicaciones mensuales, obteniendo un conjunto preliminar de 18.019 registros y 40 variables.

Sin embargo, la disponibilidad de los registros consolidados no garantiza directamente su utilización analítica. Es necesario examinar la integridad de las variables, identificar inconsistencias, evaluar los valores faltantes y establecer reglas de transformación coherentes con las características de los eventos documentados.

El nuevo enfoque territorial requiere especial atención a las variables `Comuna`, `Zona Afectada`, `Consumo` y `Distribuidoras Afectadas`, así como a la identificación de informes, sus estados administrativos y fechas relevantes.

Adicionalmente, se contempla incorporar información territorial oficial del Instituto Nacional de Estadísticas (INE), utilizando los códigos comunales y la población censada en 2024 como antecedentes para identificar las 15 comunas prioritarias del estudio.

Esta integración permitirá preparar una estructura de datos que relacione los informes de fallas con las comunas mencionadas, conservando la trazabilidad de los registros originales.

### 1.2. Objetivo general

**Preparar y validar un conjunto de datos integrado de fallas del Sistema Eléctrico Nacional y antecedentes territoriales oficiales, mediante técnicas de exploración, limpieza y transformación en Python, para sustentar el análisis comparativo de exposición operacional documentada de comunas prioritarias de Chile.**

### 1.3. Objetivos específicos

1. **Caracterizar** la estructura, composición y cobertura temporal de los registros consolidados del CEN mediante herramientas de exploración de datos.

2. **Diagnosticar** la calidad de la información, identificando valores faltantes, duplicados, inconsistencias y restricciones de las variables territoriales y operacionales.

3. **Normalizar** las variables relevantes para el análisis comunal, considerando la separación de nombres territoriales, las categorías de consumo y los tipos de datos.

4. **Integrar** los registros de fallas con información territorial oficial del INE, estableciendo correspondencias verificables entre comunas y códigos territoriales.

5. **Validar** la integridad, consistencia y trazabilidad del conjunto transformado mediante pruebas funcionales y controles de calidad.

### 1.4. Alcance metodológico

La Fase 2 comprende la obtención y lectura de los datos, su exploración inicial, el diagnóstico de calidad, la implementación de transformaciones justificadas y la validación del conjunto resultante.

Se conservará la información original y se documentarán las modificaciones realizadas durante el procesamiento.

La unidad documental será el informe de falla identificado mediante `id_falla`. Para el análisis territorial se construirá una segunda estructura que represente las asociaciones entre informes y comunas, sin interpretar cada asociación como un evento físico independiente.

Los indicadores de afectación operacional se construirán posteriormente sobre registros depurados, distinguiendo los eventos con afectación de consumo reportada de aquellos que no presentan dicha condición.

La información disponible no permite atribuir automáticamente interrupciones efectivas a todos los habitantes de una comuna ni calcular indicadores regulatorios como SAIDI y SAIFI sin antecedentes complementarios de clientes afectados y duración de sus interrupciones.

### 1.5. Criterios de reproducibilidad

El procesamiento se desarrollará mediante Python y las bibliotecas Pandas y NumPy, utilizando Jupyter Notebook como entorno de documentación y ejecución.

Las operaciones estarán organizadas en bloques funcionales, incorporando controles de entrada, validaciones de resultados y explicaciones técnicas de las transformaciones.

Los conjuntos de datos originales no serán sobrescritos. Los productos intermedios y finales se almacenarán en directorios diferenciados, manteniendo la posibilidad de reconstruir el procesamiento a partir de las fuentes documentadas.


In [1]:
# ==========================================================
# FASE 2 - OBTENCIÓN Y CARGA DEL DATASET
# PROYECTO MCDI500 - GRUPO 7
# ANÁLISIS TERRITORIAL DE FALLAS DEL SEN
# ==========================================================

from pathlib import Path

import pandas as pd
import numpy as np

from IPython.display import display


# ==========================================================
# 1. LOCALIZAR LA CARPETA PRINCIPAL DEL PROYECTO
# ==========================================================

def localizar_proyecto(ruta_inicial):

    """
    Busca la carpeta principal del proyecto
    a partir del directorio de ejecución.
    """

    inicio = Path(ruta_inicial).resolve()

    for carpeta in [inicio, *inicio.parents]:

        if (
            (carpeta / "F1").is_dir()
            and (carpeta / "src").is_dir()
            and (carpeta / "data").is_dir()
        ):

            return carpeta

    raise FileNotFoundError(
        "No se encontró la carpeta principal "
        "del proyecto MCDI500."
    )


RAIZ = localizar_proyecto(Path.cwd())


# ==========================================================
# 2. DEFINIR LA FUENTE DE DATOS
# ==========================================================

ARCHIVO_DATOS = (
    RAIZ
    / "data"
    / "processed"
    / "cen_fallas_2024_2026_eventos.csv"
)


# ==========================================================
# 3. VERIFICAR LA EXISTENCIA DEL ARCHIVO
# ==========================================================

if not ARCHIVO_DATOS.is_file():

    raise FileNotFoundError(
        f"No se encontró el dataset: {ARCHIVO_DATOS}"
    )


# ==========================================================
# 4. CARGAR LOS DATOS SIN TRANSFORMACIONES
# ==========================================================

df_cen = pd.read_csv(
    ARCHIVO_DATOS,
    encoding="utf-8-sig",
    dtype="string",
    low_memory=False
)


# ==========================================================
# 5. VERIFICAR LA ESTRUCTURA MÍNIMA
# ==========================================================

COLUMNAS_REQUERIDAS = [
    "Número",
    "id_falla",
    "Fecha Perturbación",
    "Estado",
    "Comuna",
    "Zona Afectada",
    "Consumo",
    "anio_publicacion",
    "mes_publicacion"
]

columnas_faltantes = sorted(
    set(COLUMNAS_REQUERIDAS) - set(df_cen.columns)
)

if columnas_faltantes:

    raise ValueError(
        "Faltan columnas necesarias para F2: "
        f"{columnas_faltantes}"
    )

if df_cen.empty:

    raise ValueError(
        "El conjunto de datos está vacío."
    )


# ==========================================================
# 6. MOSTRAR RESULTADOS DE CARGA
# ==========================================================

print("=" * 60)
print("FASE 2 - CARGA DEL CONJUNTO DE DATOS")
print("=" * 60)

print("\nCarpeta del proyecto:")
print(RAIZ)

print("\nArchivo utilizado:")
print(ARCHIVO_DATOS.name)

print("\nDimensiones del dataset:")
print("Registros:", df_cen.shape[0])
print("Variables:", df_cen.shape[1])

print("\nColumnas requeridas:")
print("[OK] Todas están presentes")

print("\nPrimeras observaciones:")

columnas_vista = [
    "Número",
    "Fecha Perturbación",
    "Estado",
    "Comuna",
    "Consumo",
    "Zona Afectada"
]

display(
    df_cen[columnas_vista].head(5)
)

print("\n[OK] Dataset cargado sin modificar los datos")

FASE 2 - CARGA DEL CONJUNTO DE DATOS

Carpeta del proyecto:
C:\Users\jeanp\OneDrive\00. MAGISTER CIA\M1_Programación para la Ciencia de Datos\Proyecto_MCDI500

Archivo utilizado:
cen_fallas_2024_2026_eventos.csv

Dimensiones del dataset:
Registros: 18019
Variables: 40

Columnas requeridas:
[OK] Todas están presentes

Primeras observaciones:


,Número,Fecha Perturbación,Estado,Comuna,Consumo,Zona Afectada
0,2024000001.0,2024-01-01 10:15:00,finalizado,San Clemente,consumo_no_afectado,Maule
1,2024000002.0,2024-01-01 08:20:00,finalizado,Huara,consumo_no_afectado,Tarapacá
2,2024000004.0,2024-01-01 08:20:00,finalizado,Camarones,consumo_no_afectado,Arica y Parinacota
3,2024000003.0,2024-01-01 08:20:00,finalizado,Camarones,consumo_no_afectado,Arica y Parinacota
4,2024000087.0,2024-01-09 23:30:00,finalizado,Los Andes,consumo_no_afectado,Valparaíso



[OK] Dataset cargado sin modificar los datos


In [2]:
# ==========================================================
# FASE 2 - EXPLORACIÓN INICIAL
# CARACTERIZACIÓN TERRITORIAL Y OPERACIONAL
# ==========================================================


# ==========================================================
# 1. DEFINIR VARIABLES DE INTERÉS
# ==========================================================

variables_interes = [
    "Número",
    "id_falla",
    "Comuna",
    "Zona Afectada",
    "Consumo",
    "Distribuidoras Afectadas",
    "Fecha Perturbación",
    "Fecha Efectiva Fin",
    "Estado",
    "Tipo",
    "Causa"
]

variables_disponibles = [
    columna
    for columna in variables_interes
    if columna in df_cen.columns
]


# ==========================================================
# 2. CALCULAR COMPLETITUD DE VARIABLES
# ==========================================================

diagnostico = pd.DataFrame({

    "Variable": variables_disponibles,

    "Registros completos": [
        int(df_cen[columna].notna().sum())
        for columna in variables_disponibles
    ],

    "Valores faltantes": [
        int(df_cen[columna].isna().sum())
        for columna in variables_disponibles
    ]

})

diagnostico["% faltantes"] = (
    diagnostico["Valores faltantes"]
    / len(df_cen)
    * 100
).round(2)


# ==========================================================
# 3. MOSTRAR COMPLETITUD
# ==========================================================

print("=" * 60)
print("DIAGNÓSTICO INICIAL DE VARIABLES")
print("=" * 60)

display(
    diagnostico.sort_values(
        "% faltantes",
        ascending=False
    )
)


# ==========================================================
# 4. DISTRIBUCIÓN DE CATEGORÍAS DE CONSUMO
# ==========================================================

print("\nDISTRIBUCIÓN DE LA VARIABLE CONSUMO")

display(
    df_cen["Consumo"]
    .value_counts(dropna=False)
    .rename_axis("Categoría")
    .reset_index(name="Registros")
)


# ==========================================================
# 5. IDENTIFICAR REGISTROS MULTICOMUNA
# ==========================================================

comunas_multiples = (
    df_cen["Comuna"]
    .fillna("")
    .str.contains("|", regex=False)
)

print("\nDIAGNÓSTICO TERRITORIAL")

print(
    "Registros con comuna informada:",
    int(df_cen["Comuna"].notna().sum())
)

print(
    "Registros con varias comunas:",
    int(comunas_multiples.sum())
)


# ==========================================================
# 6. VERIFICAR IDENTIFICADORES
# ==========================================================

print("\nCONTROL INICIAL DE IDENTIFICADORES")

print(
    "Identificadores faltantes:",
    int(df_cen["id_falla"].isna().sum())
)

print(
    "Identificadores repetidos:",
    int(
        df_cen["id_falla"].duplicated(keep=False).sum()
    )
)


# ==========================================================
# 7. RESULTADO DE EXPLORACIÓN
# ==========================================================

print("\n[OK] Exploración inicial finalizada")
print("[PENDIENTE] Diagnóstico exhaustivo y limpieza")

DIAGNÓSTICO INICIAL DE VARIABLES


,Variable,Registros completos,Valores faltantes,% faltantes
5,Distribuidoras Afectadas,6809,11210,62.21
7,Fecha Efectiva Fin,10805,7214,40.04
2,Comuna,16874,1145,6.35
3,Zona Afectada,16927,1092,6.06
4,Consumo,17732,287,1.59
1,id_falla,18019,0,0.00
0,Número,18019,0,0.00
6,Fecha Perturbación,18019,0,0.00
8,Estado,18019,0,0.00
9,Tipo,18019,0,0.00



DISTRIBUCIÓN DE LA VARIABLE CONSUMO


,Categoría,Registros
0,consumo_no_afectado,10901
1,consumo_regulado,3932
2,consumo_libre_regulado,1878
3,consumo_libre,1021
4,<NA>,287



DIAGNÓSTICO TERRITORIAL
Registros con comuna informada: 16874
Registros con varias comunas: 2623

CONTROL INICIAL DE IDENTIFICADORES
Identificadores faltantes: 0
Identificadores repetidos: 0

[OK] Exploración inicial finalizada
[PENDIENTE] Diagnóstico exhaustivo y limpieza
